In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_purchase_kpis AS

WITH monthly_purchase AS (

    SELECT
        DATE(DATE_TRUNC('month', date_purchase)) AS month,

        COUNT(DISTINCT purchase_order_id) AS total_orders,

        ROUND(
            SUM(amount_total),
            2
        ) AS total_spend,

        COUNT(DISTINCT supplier_id) AS active_suppliers

    FROM gold.fact_purchase_order

    WHERE date_purchase IS NOT NULL
      AND purchase_status IN ('done')

    GROUP BY DATE_TRUNC('month', date_purchase)

),

supplier_first_purchase AS (

    SELECT
        supplier_id,

        MIN(
            DATE_TRUNC('month', date_purchase)
        ) AS first_purchase_month

    FROM gold.fact_purchase_order

    WHERE purchase_status IN ('done')

    GROUP BY supplier_id

),

new_suppliers AS (

    SELECT
        first_purchase_month AS month,

        COUNT(DISTINCT supplier_id)
            AS new_suppliers

    FROM supplier_first_purchase

    GROUP BY first_purchase_month

),

final_base AS (

    SELECT
        mp.month,

        mp.total_orders,

        LAG(mp.total_orders)
            OVER (ORDER BY mp.month)
            AS prev_month_orders,

        mp.total_spend,

        LAG(mp.total_spend)
            OVER (ORDER BY mp.month)
            AS prev_month_spend,

        mp.active_suppliers,

        COALESCE(
            ns.new_suppliers,
            0
        ) AS new_suppliers_this_month

    FROM monthly_purchase mp

    LEFT JOIN new_suppliers ns
        ON mp.month = ns.month

)

SELECT
    month,

    total_orders,

    COALESCE(
        ROUND(
            (
                total_orders -
                prev_month_orders
            )
            /
            NULLIF(prev_month_orders, 0)
            * 100,
            2
        ),
        0
    ) AS orders_growth_percent,

    total_spend,

    COALESCE(
        ROUND(
            (
                total_spend -
                prev_month_spend
            )
            /
            NULLIF(prev_month_spend, 0)
            * 100,
            2
        ),
        0
    ) AS spend_growth_percent,

    active_suppliers,

    new_suppliers_this_month

FROM final_base;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_pending_purchase_approvals AS

SELECT
    COUNT(DISTINCT purchase_order_id)
        AS pending_approvals

FROM gold.fact_purchase_order

WHERE purchase_status = 'to approve';

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_purchase_spend_trend AS

WITH monthly_purchase AS (

    SELECT
        CAST(date_purchase AS DATE) AS purchase_date,

        SUM(amount_total) AS total_spend

    FROM gold.fact_purchase_order

    WHERE purchase_status IN ('done')

    GROUP BY CAST(date_purchase AS DATE)

)

SELECT
    dd.year_number,
    dd.month_number,
    dd.month_short,

    ROUND(
        COALESCE(SUM(mp.total_spend), 0),
        2
    ) AS total_spend

FROM gold.dim_date dd

LEFT JOIN monthly_purchase mp
    ON dd.full_date = mp.purchase_date

WHERE dd.year_number = YEAR(CURRENT_DATE)
  AND dd.month_number <= MONTH(CURRENT_DATE)

GROUP BY
    dd.year_number,
    dd.month_number,
    dd.month_short

ORDER BY
    dd.month_number ASC;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_top_suppliers_orders AS

SELECT
    ds.partner_name As supplier_name,

    COUNT(DISTINCT fp.purchase_order_id)
        AS total_orders

FROM gold.fact_purchase_order fp

INNER JOIN gold.dim_partner ds
    ON fp.supplier_id = ds.partner_id

WHERE fp.purchase_status IN ('done')

GROUP BY
    ds.partner_name;

In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_recent_purchase_orders AS

SELECT
    CONCAT('PO', fp.purchase_order_id)
        AS purchase_order_id,

    ds.partner_name as supplier_name,

    COUNT(fp.product_qty)
        AS items_count,

    ROUND(
        SUM(fp.amount_total),
        2
    ) AS order_amount,

    fp.purchase_status,

    CAST(fp.date_purchase AS DATE)
        AS purchase_date

FROM gold.fact_purchase_order fp

INNER JOIN gold.dim_partner ds
    ON fp.supplier_id = ds.partner_id

GROUP BY
    fp.purchase_order_id,
    ds.partner_name,
    fp.purchase_status,
    fp.date_purchase

ORDER BY fp.date_purchase DESC , fp.purchase_order_id DESC;

In [0]:

Select * 
from smart_odoo.gold.vw_purchase_kpis 
ORDER BY month DESC LIMIT 1;
---------------------------------------------
Select * 
from smart_odoo.gold.vw_pending_purchase_approvals;
---------------------------------------------
Select *
from smart_odoo.gold.vw_purchase_spend_trend;
---------------------------------------------
Select * 
from gold.vw_top_suppliers_orders 
ORDER BY total_orders DESC LIMIT 5;
---------------------------------------------
Select * 
from  gold.vw_recent_purchase_orders
limit 5; 
